# Sparse DSRG for H4: reference-normal vs bare physical-rank truncation

This notebook compares two sparse single-reference DSRG variants for linear H4/STO-3G at `s = 5`.

- **Reference-normal DSRG(n)**: after every BCH commutator, transform to determinant-reference normal order and retain normal-ordered terms through many-body rank `n`.
- **Bare physical-rank DSRG(n)**: do not determinant-normal-order the transformed operator. Instead, truncate the raw `SparseOperator` by physical operator rank and evaluate the DSRG energy/residuals by explicit projections, `<Phi|Hbar|Phi>` and `<Phi_mu|Hbar|Phi>`.

`SparseOperator` still stores strings in its canonical creation-before-annihilation form and its products include the usual fermionic contractions. The second variant avoids the new determinant-reference normal-ordering layer in the DSRG loop.


In [ ]:
import contextlib
import io
import itertools
import math
import time

import numpy as np

from forte2 import RHF, System
from forte2.helpers import logger
from forte2.lib.det import Determinant, hilbert_space
from forte2.lib.sparse_ops import (
    NormalOrderedSparseOperator,
    SparseOperator,
    SparseState,
    normal_order,
    overlap,
    sparse_operator,
    sparse_operator_hamiltonian,
)

logger.set_verbosity_level(0)

SCREEN = 1.0e-12
FLOW_PARAM = 5.0
SPACING = 0.74
BASIS = "sto-3g"


In [ ]:
def build_linear_h_sparse_hamiltonian(natoms, r=0.74, basis="sto-3g"):
    xyz = "\n".join(f"H 0.0 0.0 {i * r:.12f}" for i in range(natoms))

    with contextlib.redirect_stdout(io.StringIO()):
        system = System(
            xyz=xyz,
            basis_set=basis,
            minao_basis_set=None,
            cholesky_tei=True,
            cholesky_tol=1.0e-12,
        )
        rhf = RHF(charge=0, e_tol=1.0e-10, d_tol=1.0e-8)(system)
        rhf.run()

    C = rhf.C[0]
    hcore_mo = np.einsum("pq,pi,qj->ij", system.ints_hcore(), C, C, optimize=True)
    eri_mo = system.fock_builder.two_electron_integrals_block(C)
    ham = sparse_operator_hamiltonian(system.nuclear_repulsion, hcore_mo, eri_mo)
    return system, rhf, ham, np.array(rhf.eps[0])


def fci_energy_from_sparse_operator(ham, nmo, na, nb):
    dets = hilbert_space(nmo, na, nb)
    index = {det: i for i, det in enumerate(dets)}
    hmat = np.zeros((len(dets), len(dets)), dtype=complex)

    for j, ket_det in enumerate(dets):
        hket = ham @ SparseState({ket_det: 1.0})
        for bra_det, coeff in hket.items():
            i = index.get(bra_det)
            if i is not None:
                hmat[i, j] = coeff

    return np.linalg.eigvalsh(hmat)[0].real, len(dets)


In [ ]:
def canonical_excitation_string(cre_modes, ann_modes):
    def token(mode, creation):
        orbital, spin = mode
        return f"{orbital}{spin}{'+' if creation else '-'}"

    alpha_cre = sorted([m for m in cre_modes if m[1] == "a"], key=lambda x: x[0])
    beta_cre = sorted([m for m in cre_modes if m[1] == "b"], key=lambda x: x[0])
    beta_ann = sorted([m for m in ann_modes if m[1] == "b"], key=lambda x: x[0], reverse=True)
    alpha_ann = sorted([m for m in ann_modes if m[1] == "a"], key=lambda x: x[0], reverse=True)

    tokens = [token(m, True) for m in alpha_cre]
    tokens += [token(m, True) for m in beta_cre]
    tokens += [token(m, False) for m in beta_ann]
    tokens += [token(m, False) for m in alpha_ann]
    return "[" + " ".join(tokens) + "]"


def enumerate_spin_conserving_excitations(nspatial, nocc, eps, ref, max_excitation_rank):
    occ = [(i, spin) for i in range(nocc) for spin in ("a", "b")]
    virt = [(a, spin) for a in range(nocc, nspatial) for spin in ("a", "b")]
    highest_rank = min(max_excitation_rank, len(occ), len(virt))
    ref_state = SparseState({ref: 1.0})
    excitations = []

    for rank in range(1, highest_rank + 1):
        for ann in itertools.combinations(occ, rank):
            ann_spins = sorted(spin for _, spin in ann)
            for cre in itertools.combinations(virt, rank):
                if ann_spins != sorted(spin for _, spin in cre):
                    continue

                label = canonical_excitation_string(cre, ann)
                op = sparse_operator(label, 1.0)
                state = op @ ref_state
                if len(list(state.items())) != 1:
                    raise RuntimeError(f"Excitation {label} did not produce one determinant")

                denom = sum(eps[i] for i, _ in ann) - sum(eps[a] for a, _ in cre)
                excitations.append(
                    {
                        "rank": rank,
                        "label": label,
                        "op": op,
                        "state": state,
                        "denom": denom,
                    }
                )

    return excitations


def make_cluster_operator(excitations, amplitudes):
    T = SparseOperator()
    for ex, amp in zip(excitations, amplitudes):
        if abs(amp) > SCREEN:
            T += ex["op"] * complex(amp)
    return T


def regularized_denominator(denom, s):
    return (1.0 - math.exp(-s * denom * denom)) / denom


def sparse_operator_norm(op):
    return math.sqrt(sum(abs(coeff) ** 2 for _, coeff in op)) if len(op) else 0.0


## Reference-normal DSRG(n)

This is the determinant-reference normal-ordered variant from the first notebook. The scalar and residuals are read as coefficients of the reference-normal operator. The cluster operator `T` is also assembled in this reference-normal basis and expanded to `SparseOperator` only for commutators.


In [ ]:
def normal_key_and_phase(spop, ref):
    no_op = normal_order(spop, ref, SCREEN)
    items = [(term, coeff) for term, coeff in no_op if abs(coeff) > 1.0e-10]
    if len(items) != 1:
        raise RuntimeError(
            "Expected one normal-ordered term, got "
            + str([(term.str(ref), coeff) for term, coeff in items])
        )
    return items[0]


def physical_coeff(no_op, key, phase):
    return no_op.coefficient(key) / phase


def make_normal_ordered_cluster_operator(excitations, amplitudes, ref):
    T_no = NormalOrderedSparseOperator(ref)
    for ex, amp in zip(excitations, amplitudes):
        if abs(amp) > SCREEN:
            T_no.add(ex["key"], complex(amp) * ex["phase"])
    return T_no


def excitation_normal_keys(excitations, ref):
    keyed = []
    for ex in excitations:
        key, phase = normal_key_and_phase(ex["op"], ref)
        keyed.append({**ex, "key": key, "phase": phase})
    return keyed


def truncate_reference_normal_operator(op, ref, truncation_rank):
    return normal_order(op, ref, SCREEN, max_rank=truncation_rank).to_sparse_operator(SCREEN)


def bch_hbar_reference_normal(ham, A, ref, truncation_rank, max_comm=20, comm_thresh=1.0e-12):
    hbar = truncate_reference_normal_operator(ham, ref, truncation_rank)
    nested = hbar
    commutator_norms = []

    for ncomm in range(1, max_comm + 1):
        nested = nested.commutator(A)
        nested = truncate_reference_normal_operator(nested, ref, truncation_rank)
        contribution = nested * (1.0 / math.factorial(ncomm))
        hbar += contribution

        norm = sparse_operator_norm(contribution)
        commutator_norms.append(norm)
        if norm < comm_thresh:
            break

    return normal_order(hbar, ref, SCREEN, max_rank=truncation_rank), commutator_norms


def solve_reference_normal_dsrg(
    ham,
    ref,
    excitations,
    truncation_rank,
    flow_param=FLOW_PARAM,
    max_iter=80,
    e_conv=1.0e-12,
    r_conv=1.0e-10,
    max_comm=20,
):
    excitations = excitation_normal_keys(excitations, ref)
    ham_no = normal_order(ham, ref, SCREEN, max_rank=truncation_rank)
    h0 = np.array(
        [physical_coeff(ham_no, ex["key"], ex["phase"]) for ex in excitations],
        dtype=complex,
    )
    amplitudes = np.array(
        [h0[k] * regularized_denominator(ex["denom"], flow_param) for k, ex in enumerate(excitations)],
        dtype=complex,
    )

    identity_key, identity_phase = normal_key_and_phase(sparse_operator("[]", 1.0), ref)
    history = []
    previous_energy = None
    t0 = time.perf_counter()

    for iteration in range(max_iter + 1):
        T_no = make_normal_ordered_cluster_operator(excitations, amplitudes, ref)
        T = T_no.to_sparse_operator(SCREEN)
        A = T - T.adjoint()
        hbar_no, commutator_norms = bch_hbar_reference_normal(
            ham, A, ref, truncation_rank=truncation_rank, max_comm=max_comm
        )
        energy = physical_coeff(hbar_no, identity_key, identity_phase).real
        hbar_offdiag = np.array(
            [physical_coeff(hbar_no, ex["key"], ex["phase"]) for ex in excitations],
            dtype=complex,
        )
        new_amplitudes = np.array(
            [
                (hbar_offdiag[k] + ex["denom"] * amplitudes[k])
                * regularized_denominator(ex["denom"], flow_param)
                for k, ex in enumerate(excitations)
            ],
            dtype=complex,
        )

        rms_update = float(np.linalg.norm(new_amplitudes - amplitudes))
        delta_energy = 0.0 if previous_energy is None else energy - previous_energy
        history.append(
            {
                "iteration": iteration,
                "energy": energy,
                "delta_energy": delta_energy,
                "rms_update": rms_update,
                "ncomm": len(commutator_norms),
            }
        )

        if previous_energy is not None and abs(delta_energy) < e_conv and rms_update < r_conv:
            return energy, amplitudes, history, time.perf_counter() - t0

        amplitudes = new_amplitudes
        previous_energy = energy

    raise RuntimeError(f"Reference-normal DSRG({truncation_rank}) did not converge")


## Bare physical-rank DSRG(n)

The solver below never calls `normal_order`. It keeps raw `SparseOperator` terms with physical rank `<= n`, where the rank is half the number of second-quantized operators in a number-conserving string. Because the operator is not reference-normal ordered, coefficients of identity/excitation strings are not the right scalar/residuals. We therefore compute them by projection on the reference determinant and excited determinants.

In [ ]:
def sparse_body_rank(sqop):
    count = sqop.count()
    if count % 2 != 0:
        raise ValueError(f"Only particle-number-conserving strings are supported: {sqop}")
    return count // 2


def truncate_physical_rank(op, max_rank, screen_thresh=SCREEN):
    truncated = SparseOperator()
    for sqop, coeff in op:
        if abs(coeff) > screen_thresh and sparse_body_rank(sqop) <= max_rank:
            truncated.add(sqop, coeff)
    return truncated


def bch_hbar_bare_physical_rank(ham, T, truncation_rank, max_comm=20, comm_thresh=1.0e-12):
    A = T - T.adjoint()
    hbar = truncate_physical_rank(ham, truncation_rank)
    nested = hbar
    commutator_norms = []

    for ncomm in range(1, max_comm + 1):
        nested = nested.commutator(A)
        nested = truncate_physical_rank(nested, truncation_rank)
        contribution = nested * (1.0 / math.factorial(ncomm))
        hbar += contribution

        norm = sparse_operator_norm(contribution)
        commutator_norms.append(norm)
        if norm < comm_thresh:
            break

    return truncate_physical_rank(hbar, truncation_rank), commutator_norms


def solve_bare_physical_rank_dsrg(
    ham,
    ref,
    excitations,
    truncation_rank,
    flow_param=FLOW_PARAM,
    max_iter=80,
    e_conv=1.0e-12,
    r_conv=1.0e-10,
    max_comm=20,
):
    ref_state = SparseState({ref: 1.0})
    h0_state = truncate_physical_rank(ham, truncation_rank) @ ref_state
    h0 = np.array([overlap(ex["state"], h0_state) for ex in excitations], dtype=complex)
    amplitudes = np.array(
        [h0[k] * regularized_denominator(ex["denom"], flow_param) for k, ex in enumerate(excitations)],
        dtype=complex,
    )

    history = []
    previous_energy = None
    t0 = time.perf_counter()

    for iteration in range(max_iter + 1):
        T = make_cluster_operator(excitations, amplitudes)
        hbar, commutator_norms = bch_hbar_bare_physical_rank(
            ham, T, truncation_rank=truncation_rank, max_comm=max_comm
        )
        hbar_ref = hbar @ ref_state
        energy = overlap(ref_state, hbar_ref).real
        hbar_offdiag = np.array(
            [overlap(ex["state"], hbar_ref) for ex in excitations],
            dtype=complex,
        )
        new_amplitudes = np.array(
            [
                (hbar_offdiag[k] + ex["denom"] * amplitudes[k])
                * regularized_denominator(ex["denom"], flow_param)
                for k, ex in enumerate(excitations)
            ],
            dtype=complex,
        )

        rms_update = float(np.linalg.norm(new_amplitudes - amplitudes))
        delta_energy = 0.0 if previous_energy is None else energy - previous_energy
        history.append(
            {
                "iteration": iteration,
                "energy": energy,
                "delta_energy": delta_energy,
                "rms_update": rms_update,
                "ncomm": len(commutator_norms),
                "n_terms": len(hbar),
            }
        )

        if previous_energy is not None and abs(delta_energy) < e_conv and rms_update < r_conv:
            return energy, amplitudes, history, time.perf_counter() - t0

        amplitudes = new_amplitudes
        previous_energy = energy

    raise RuntimeError(f"Bare physical-rank DSRG({truncation_rank}) did not converge")


## H4 calculation

The molecule is linear H4 with `R(H-H) = 0.74 Angstrom`, STO-3G, closed-shell RHF orbitals, and `s = 5`.

In [ ]:
system, rhf, ham, eps = build_linear_h_sparse_hamiltonian(4, r=SPACING, basis=BASIS)
reference = Determinant("2" * rhf.na + "0" * (rhf.nmo - rhf.na))
E_fci, ndet = fci_energy_from_sparse_operator(ham, rhf.nmo, rhf.na, rhf.nb)

print(f"RHF energy:       {rhf.E:.15f} Eh")
print(f"FCI energy:       {E_fci:.15f} Eh")
print(f"FCI determinants: {ndet}")
print(f"Reference:        {reference.str(rhf.nmo)}")
print(f"Hamiltonian terms:{len(ham)}")


In [ ]:
rank_sweep = []

for rank in (2, 3, 4):
    excitations = enumerate_spin_conserving_excitations(
        rhf.nmo,
        rhf.na,
        eps,
        reference,
        max_excitation_rank=rank,
    )

    E_normal, amps_normal, hist_normal, time_normal = solve_reference_normal_dsrg(
        ham,
        reference,
        excitations,
        truncation_rank=rank,
        flow_param=FLOW_PARAM,
    )
    E_bare, amps_bare, hist_bare, time_bare = solve_bare_physical_rank_dsrg(
        ham,
        reference,
        excitations,
        truncation_rank=rank,
        flow_param=FLOW_PARAM,
    )

    rank_sweep.append(
        {
            "rank": rank,
            "n_amplitudes": len(excitations),
            "E_normal": E_normal,
            "E_bare": E_bare,
            "normal_iterations": len(hist_normal),
            "bare_iterations": len(hist_bare),
            "normal_seconds": time_normal,
            "bare_seconds": time_bare,
            "normal_error_mEh": 1000.0 * (E_normal - E_fci),
            "bare_error_mEh": 1000.0 * (E_bare - E_fci),
        }
    )

print(
    f"{'n':>2s} {'amps':>5s} {'E normal / Eh':>20s} {'E bare / Eh':>20s} "
    f"{'normal-FCI / mEh':>18s} {'bare-FCI / mEh':>16s} "
    f"{'it n/b':>9s} {'sec n/b':>15s}"
)
for row in rank_sweep:
    print(
        f"{row['rank']:2d} {row['n_amplitudes']:5d} "
        f"{row['E_normal']:20.15f} {row['E_bare']:20.15f} "
        f"{row['normal_error_mEh']:18.9f} {row['bare_error_mEh']:16.9f} "
        f"{row['normal_iterations']:4d}/{row['bare_iterations']:<4d} "
        f"{row['normal_seconds']:7.2f}/{row['bare_seconds']:<7.2f}"
    )


Representative output from this machine:

| n | amplitudes | reference-normal DSRG(n) / Eh | bare physical-rank DSRG(n) / Eh | normal-FCI / mEh | bare-FCI / mEh |
| ---: | ---: | ---: | ---: | ---: | ---: |
| 2 | 26 | -2.140264303594847 | -2.143831326588781 | -1.374388672 | -4.941411666 |
| 3 | 34 | -2.138895129861726 | -2.139015493226138 | -0.005214939 | -0.125578303 |
| 4 | 35 | -2.138889864326098 | -2.138889914914326 | 0.000050597 | 0.000000009 |

The two truncation definitions are not equivalent at low rank. The bare physical-rank variant keeps occupied number-operator structure instead of contracting it into reference-normal lower-body terms, so its rank-2 and rank-3 energies differ from the reference-normal DSRG(n) energies. At rank 4 for H4, the bare physical-rank truncation is effectively exact for the available electron/orbital space and lands on the FCI energy to numerical precision.